In [ ]:
import pulp as plp

In [ ]:
# Constants
max_tables = 5
max_table_size = 4
guests = "A B C D E F G I J K L M N O P Q R".split()


# Define happiness function - note this can be non-linear
def happiness(table: tuple[str, ...]) -> float:
    return abs(ord(table[0]) - ord(table[-1]))


# All possible ways of seating up to max_table_size guests at a table
possible_tables = list(plp.allcombinations(guests, max_table_size))

# Decision variable indicating which possible table is used
tables = plp.LpVariable.dicts(
    "table", possible_tables, lowBound=0, upBound=1, cat=plp.LpInteger
)

# Define problem
seating_model = plp.LpProblem("wedding_seating_model", plp.LpMaximize)

# Objective function
seating_model += plp.lpSum([happiness(t) * tables[t] for t in possible_tables])

# Constraints
seating_model += (
    plp.lpSum([tables[t] for t in possible_tables]) <= max_tables,
    "maximum_number_of_tables",
)

for guest in guests:
    seating_model += (
        plp.lpSum([tables[t] for t in possible_tables if guest in t]) == 1,
        f"must_seat_{guest}",
    )

seating_model.solve(solver=plp.PULP_CBC_CMD(msg=False))

print(f"Optimal tables out of all {len(possible_tables)} possible tables:")
for table in possible_tables:
    if tables[table].value() == 1:
        print(table)